# Working with Continuous Treatment Models in `causalprog`

This example aims to illustrate the functionality of the `causalprog.graph.ricardo` submodule.
In particular; we will cover the purposes of the major functions within this package, any additional setup the user will need to conduct a-priori, and how to interact with the model that is constructed and estimate the causal bounds.

Our example will be grounded in the framework of continuous treatment models, and we adopt the language and notation of [the description that can be found in the documentation](https://github-pages.ucl.ac.uk/causalprog/theory/continuous-treatments/).

## Imports

To keep the actual workflow of the notebook clean, we will conduct all the imports that we require in the cell below.

In [ ]:
import jax
import jax.numpy as jnp
import jax.random as jrn

from causalprog.graph.ricardo import (
    MLPAlias,
    ModelParam,
    build_causal_response_function,
    build_loss_function,
    build_regression_function,
    example_model,
)
from causalprog.quadrature import UniformWeightMonteCarloGaussianQuadrature as UWMCGQuad
from causalprog.solvers import augmented_lagrangian, stochastic_gradient_descent

jax.config.update("jax_enable_x64", val=True)

rng_key = jrn.key(seed=0)

<!-- TODO: X-ref to docs pages, also name of submodule shouldn't be ricardo! -->

Within these imports are:

- `jax`, which the package will ultimately use for efficient computations on both CPU and GPU systems.
  `jax.numpy` is a submodule that provides `numpy`-like operations, and the `jax.config.update` command enables `float64` precision operations as opposed to the default `float32`.
  Similarly, `jax.random` provides random number generation support, which we'll need when conducting Monte-Carlo integration.
- `causalprog.graph.ricardo` contains helper functions for interacting with continuous treatment models as described in [the documentation](https://github-pages.ucl.ac.uk/causalprog/theory/continuous-treatments/).
  We will be discussing these functions in more detail as we encounter them in our workflow.
  The `MLPAlias` and `ModelParam` imports are for type-hinting purposes, and describe the format for MLPs and model parameters that `causalprog` is expecting.
- `causalprog.quadrature` provides quadrature rules for approximating stochastic integrals.
- `causalprog.solvers` provides us with a few JAX-friendly ways to solve the resulting optimisation problem involving the causal response function.

## Setting up the Example Problem

For this example we will look to solve a problem that reduces to a quadratically-constrained linear objective function, [as described here](https://github-pages.ucl.ac.uk/causalprog/theory/reduce-to-linear-example/).
However we will highlight the points in the workflow that can be - and are expected to be - generalised for actual use-cases.
The example linked to above is just being used in the interest of having a lightweight example that can be run on-demand, and provide an analytic answer.

### Building the Model

We can build the model using the `example_model` function we imported from `causalprog.graph.ricardo`, by supplying the maps between random variables (and implicitly the parameters that characterise them) to this function.

To this end, we begin by constructing the mapping functions that this problem entails.

In [ ]:
def mlps_for_example(d_z: int, k_len: int) -> dict[str, MLPAlias]:
    r"""
    Construct MLP-stand-ins used in this problem.

    - $f_m$ returns 0 so that the sigmoid it's passed into always returns 0.5.
    - $f_r$ just returns a 1-vector of appropriate length.
    - $f_{pi}$ also just returns a 1-vector.
    - $g(x, z, l) = -x \mathbb{I}$ where $\mathbb{I}$ is the $\mathbb{R}^{d_z}$ unit
        vector with identical elements.
    - $f_y$ is defined by `f_y`, above.
    """

    def f_m(*args, **kwargs) -> jax.Array:
        """Note that this results in sigmoid(f_m) = 0.5 always."""
        return 0.0

    def f_r(*args, **kwargs) -> jax.Array:
        """Results in the 1-vector in R^d_z after passing through tanh."""
        return jnp.full((d_z,), float("inf"))

    def f_pi(*args, **kwargs) -> jax.Array:
        """Theoretically irrelevant as it will be softmax'd."""
        return jnp.ones((k_len,))

    def g(xzl: dict[str, jax.Array], _: ModelParam) -> jax.Array:
        """Form ensures that m_y^T g gives us a mean of -x/2."""
        return -xzl["x"] * jnp.ones((d_z,)) / jnp.sqrt(d_z)

    def f_y(u_yxl: dict[str, jax.Array], theta_y: ModelParam) -> jax.Array:
        r"""$f_Y(u_y, x, l; \theta_Y) = \frac{\theta_Y}{l}(u_y - x)^2$."""
        return (theta_y / u_yxl["l"]) * (u_yxl["u_y"] - u_yxl["x"]) ** 2

    return {"f_r": f_r, "f_m": f_m, "f_pi": f_pi, "g": g, "f_y": f_y}

Note that in almost all practical examples, the functions created here will not be simple deterministic expressions.

The functions `f_m`, `f_r`, `f_pi`, `f_y` are expected to be multilayer perceptrons as described in [TODO: link Sam's MLP descriptions](FIXME).
The function `g` is expected to be the [inverse map of a pre-trained normalising flow feed-forward network](https://github-pages.ucl.ac.uk/causalprog/theory/continuous-treatments/#model-for-treatment-x).
Though in practice, they can be any `jit`-able function(s) of two arguments;

- The first argument being a dictionary whose keys are the names of the dependent nodes / RVs that the function takes as direct inputs.
  The values of each key should be the corresponding input values for that RV.
- The second argument should be the model parameters $\theta_{\alpha}$ that parametrise the function (or predictive model described by the function).
  This will be a subset of the parameters over which the causal response function will be estimated.

Hence why, in this case, we can simply take them as simple deterministic functions parameterised by their respective parameter values.

In any event, once we have our mapping functions defined, we can build the model.

In [ ]:
# Set the hyper-parameters of the problem.
d_z = 5
k_len = 10

# Build the MLPs for the problem.
# Note that there is no need in-general to have each MLP saved to a key in
# a dictionary - they can be defined or constructed inline if preferred.
mlps = mlps_for_example(d_z=d_z, k_len=k_len)

# FIXME: Nicer means of building this plz
# Build model, manual attachment to nodes for now...
treatment_model = example_model(
    z_len=d_z,
    compute_u_x=mlps["g"],
    compute_u_y=mlps["f_pi"],
    compute_x=None,
    compute_phi_x=None,
    compute_y=mlps["f_y"],
)
treatment_model.get_node("u_y").f_r = mlps["f_r"]
treatment_model.get_node("u_y").f_m = mlps["f_m"]

`treatment_model` is a [`causalprog.graph.Graph` instance](https://github-pages.ucl.ac.uk/causalprog/api/#causalprog.graph.graph.Graph) that now describes our particular treatment model.
The various mapping functions have each been attached to the appropriate sites in the model, and methods from the `algorithms` sub-module can be applied to the model just like any other `Graph`.

In theory, at this point the set of learnt parameters $\theta_X$ is known given how $g$ is defined.
In this example, we have taken $g$ to be independent of $\theta_X$, but for the sake of consistency with the more general case we will assign a placeholder value for $\theta_X$ below, that will need to be passed into the various `causalprog` constructor functions. 

In [ ]:
learnt_theta_x = jnp.atleast_1d(0.0)

### The Regression Function

Now that we have our `treatment_model`, we can delegate building the [regression function $r$](https://github-pages.ucl.ac.uk/causalprog/theory/continuous-treatments/#model-for-outcome-y), [loss function $B$](https://github-pages.ucl.ac.uk/causalprog/theory/continuous-treatments/#learn-initialiser), and [causal response $d$](https://github-pages.ucl.ac.uk/causalprog/theory/continuous-treatments/#model-for-outcome-y) to `causalprog`.

To build $r$ and $d$, we are required to perform a stochastic integral, and thus must specify the quadrature scheme that we want `causalprog` to employ when it does this.
To that end; we simply need to pick a compatible quadrature class from `causalprog.quadrature` and the number of sample points we want to use when we estimate the integral.
Due to the nature of random number generation in JAX, we must also specify the PRNG key that is to be used when generating samples when we initialise the quadrature scheme too.

In [ ]:
# A general rule of thumb for Monte-Carlo integration is that the error in
# the estimation of the integral is inversely proportional to the square-root
# of the number of sample points used, all other factors being constant.
n_sample_pts = 1_000_000

quad_method = UWMCGQuad(n_points=n_sample_pts, rng_key=rng_key)

regression_function = build_regression_function(
    treatment_model,
    theta_x=learnt_theta_x,
    quadrature=quad_method,
)

`regression_function` is a Python function of two inputs, which are identical in interpretation to that of the various $f_{\alpha}$ functions and $g$.
Relating the mathematical notation to the programming syntax, 

$$
r(x, z, l; \theta) \quad\text{where}\quad \theta = \theta_Y \cup \theta_{\pi} \cup \theta_m \cup \theta_r,
$$

is equivalent to

```python
regression_function(
  {"x": x, "z": z, "l" l},
  {
    "theta_y": theta_y,
    "theta_pi": theta_pi,
    "theta_m": theta_m,
    "theta_r": theta_r
  }
)
```

`build_regression_function` can be supplied two keyword arguments; `domain_lower_bound` and `domain_upper_bound`, which will restrict the interval over which the integral is performed.
These keyword arguments should only be used in cases where some information about the form of the integrand is known a-priori, and you believe that restricting the domain of integration will still yield a good approximation to the result.

### The Loss Function and Learnt Initialiser

Building the loss function and determining the learnt initialiser $\theta^{\star}$ can also be done in a couple of steps, once we have the `regression_function` ready to go and the set of points $\hat{r}_i$ that have been [obtained from estimating $r$ using a set of training points $\mathcal{D}_{train}$](https://github-pages.ucl.ac.uk/causalprog/theory/continuous-treatments/#learning-and-querying).

Again, since this example relies on an analytic answer, we will be setting $\hat{r} = 0$, and only making use of a single evaluation point $\mathcal{D}_{eval} = \{ (\tilde{x}, \tilde{z}, \tilde{l}) \}$.
But we define variables for these variables to preserve the generality of the syntax in this example.

In [ ]:
# Note that, if D_eval consisted of n_eval training points,
# then these variables should be 1D-arrays of n_eval elements.
# The i-th point in D_eval would then be the point formed from
# taking the i-th value of each of these arrays.
x_tilde = 0.1
z_tilde = 2.0
l_tilde = 1.0

D_eval = {
    "x": jnp.atleast_1d(x_tilde),
    "z": jnp.atleast_1d(z_tilde),
    "l": jnp.atleast_1d(l_tilde),
}
# In general, this is a 1D vector of n_eval points
r_hat_i = jnp.atleast_1d(0.0)

With these ingredients, the loss function can be built.

In [ ]:
loss_function = build_loss_function(regression_function, D_eval, r_hat_i)

Similarly to `build_regression_function`, `build_loss_function` returns a Python callable that evaluates the loss function $B$.

$$
B(\theta) \quad\text{where}\quad \theta = \theta_Y \cup \theta_{\pi} \cup \theta_m \cup \theta_r,
$$

is equivalent to

```python
loss_function(
  {
    "theta_y": theta_y,
    "theta_pi": theta_pi,
    "theta_m": theta_m,
    "theta_r": theta_r
  }
)
```

Note that `D_eval` has been baked into the construction of $B$.
If the evaluation points (or training points) are later changed, one will have to reconstruct $B$ via `build_loss_function` again.

Now that we can evaluate $B$, we can run our favourite optimisation algorithm over its argument to obtain the learnt initialiser $\theta^{\star}$.

In [ ]:
initial_guess = {
    "theta_y": 0.5,
    "theta_pi": 0.0,
    "theta_m": 0.0,
    "theta_r": 0.0,
}
sgd_result = stochastic_gradient_descent(
    loss_function,
    initial_guess,
)

learnt_initialiser = sgd_result.fn_args
print(f"Learnt theta_y (should be close to 0): {learnt_initialiser['theta_y']:.5e}")
minimised_loss = sgd_result.obj_val
print(f"Minimum loss: {minimised_loss:.5e}")

Note that it is necessary to pass in the keys for `theta_pi`, `theta_m`, and `theta_r` even though we know that the model is independent of these values (since we have chosen to employ a simple, analytically-tractable example). JAX should quickly realise that the loss function is independent of these parameters however, and as such they should not deviate far from their initial values.

### The Causal Response

The causal response function $d$ can be created from the `treatment_model` in a similar fashion to the regression function.
Since it involves an integral, we will need to specify a quadrature scheme again. 

In [ ]:
# Whilst there is no obligation to use the same quadrature scheme for
# both the regression function and response function, in this example we
# will do just that.
response_function = build_causal_response_function(treatment_model, quad_method)

`response_function` is another Python callable of two inputs, just like the format of $r$ and the various mappings $f_{\alpha}$ and $g$.

$$
d(x, l; \theta) \quad\text{where}\quad \theta = \theta_Y \cup \theta_{\pi} \cup \theta_m \cup \theta_r,
$$

is equivalent to

```python
response_function(
  {"x": x, "l" l},
  {
    "theta_y": theta_y,
    "theta_pi": theta_pi,
    "theta_m": theta_m,
    "theta_r": theta_r
  }
)
```

## Finding Bounds on the Causal Response

Now that all the components of our problem are setup, we can look to determine bounds on our causal response function.
As a reminder, we are interested in solving

$$
\min/\max_{\theta} d(x, l; \theta)
\quad\text{subject to}\quad
B(\theta) \leq B(\theta^\star) + \epsilon
$$

for some user-prescribed tolerance $\epsilon$.

In [ ]:
# The example we follow defines a variable delta = sqrt(epsilon),
# which makes the analytic solution a bit nicer to look at when written down.
# In general, one just sets a value of epsilon here.
delta = 0.5
epsilon = delta**2

# We also need to specify the values of x and l that are passed to the
# response function, that we will be solving at.
xl_to_solve_at = {"x": 2.0, "l": 2.0}

`causalprog` provides an implementation of both an augmented Lagrangian method, and the more general Penalty method, in [the `causalprog.solvers` submodule](https://github-pages.ucl.ac.uk/causalprog/api/#causalprog.solvers).

In [ ]:
# ! This cell takes ~75 seconds to run on a laptop purchased in 2023.

# Initial guess provided to solver can be anything user-defined, but
# the learnt initialiser is generally a good place to start.
opt_initial_guess = learnt_initialiser


# Constraints assume the form b(theta) < epsilon,
# so we need to slightly offset our loss function.
def constraint(theta: ModelParam) -> jax.Array:
    """B(theta) - B(theta^*) < epsilon."""
    return loss_function(theta) - minimised_loss


min_result = augmented_lagrangian(
    lambda theta: response_function(xl_to_solve_at, theta),
    opt_initial_guess,
    constraint,
    bounds_epsilon=epsilon,
    maxiter=6,
    update_mu=lambda mu: 3.0 * mu,
    update_learning_rate=lambda lr, mu, _: lr / jnp.sqrt(mu),
)
max_result = augmented_lagrangian(
    lambda theta: response_function(xl_to_solve_at, theta),
    opt_initial_guess,
    constraint,
    bounds_epsilon=epsilon,
    max_or_min="max",
    maxiter=6,
    update_mu=lambda mu: 3.0 * mu,
    update_learning_rate=lambda lr, mu, _: lr / jnp.sqrt(mu),
)

As with most penalty methods, the initial guess and how the step size and penalty factor increase (and in what proportions) needs to be appropriately set based on our understanding of the problem we're attempting to solve.

In this example we can afford to be quite brazen in our approach; only requiring a small number of iterations and fairly large penalty increases and learning-rate decreases, to obtain a good approximation to the analytic answer.

The analytic values we are expecting for $\theta_Y$ are

$$
\theta_Y = \pm \frac{4 \delta \tilde{l}}{3 (1 + 3\tilde{x}^2)},
$$

with the positive solution in the maximisation case.
Our example is independent of the other parameters, so they should all remain close to their given initial values.

In [ ]:
theta_y_analytic = delta * l_tilde * (4 / 3) / (1.0 + 3.0 * x_tilde**2)

print(
    "Minimising |",
    f"solver: {min_result.fn_args['theta_y']:.5e}",
    f"analytic: {-theta_y_analytic:.5e}",
)
print(
    "Maximising |",
    f"solver: {max_result.fn_args['theta_y']:.5e}",
    f"analytic: {theta_y_analytic:.5e}",
)
print(
    "Estimates on the response function bounds:",
    f"{min_result.obj_val:.5e}",
    f"{max_result.obj_val:.5e}",
)